<a href="https://colab.research.google.com/github/mx-oscar-hdez/Proyectos/blob/main/Mx_Oscar_Hdez_Hands_On_Dise%C3%B1o_de_Agentes_Conversacionales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: DISEÑO DE AGENTES CONVERSACIONALES**

Una vez vista la masterclass ***Diseño de Agentes Conversacionales***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.


De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/18wkM2LSfY39VZ2GBdcPCY2Qw0EzewJsx?usp=drive_link).

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**


#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [22]:
# Leer credenciales desde Colab Secrets
from google.colab import userdata

VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")
WHATSAPP_TOKEN = userdata.get("WHATSAPP_TOKEN")
PHONE_NUMBER_ID = userdata.get("PHONE_NUMBER_ID")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")

print("Credenciales cargadas correctamente")


Credenciales cargadas correctamente


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [2]:
# Instalar Ollama
!apt-get -y update -qq
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 39 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (630 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controllin

In [3]:
# Iniciar el servidor de Ollama
import subprocess, time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

!pip install ollama --quiet
import ollama

!ollama pull llama3.1
print("Ollama listo.")


Ollama listo.


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [4]:
# Instalar dependencias
!pip install -q "llama-stack[starter]" llama-stack-client
!pip install -q "mcp<2.0" --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 5.4 MB/s eta 0:00:00
  

In [5]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado
import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(
    os.path.dirname(llama_stack.__file__),
    "distributions",
    "starter",
    "config.yaml"
)

with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:latest",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:latest",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)

if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


In [6]:
import requests

try:
    r = requests.get("http://127.0.0.1:8321/v1/models")
    print("Llama Stack:", "OK" if r.status_code == 200 else r.status_code)
except Exception as e:
    print("Llama Stack no disponible:", repr(e))

Llama Stack: OK


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [7]:
# Configurar el servidor del webhook
from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()

In [8]:
# Definir función de normalización de número
def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que guarda, por cada número de teléfono, el historial reciente de mensajes — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación. Esta es una implementación manual y simplificada del concepto; la lectura del Tema menciona **Llama Stack** como el framework que estandariza esta pieza (junto con herramientas y seguridad) para no reconstruirla en cada proyecto. **Nota:** el patrón exacto de integración de Llama Stack debe confirmarse contra su documentación vigente antes de grabar o dar la sesión en vivo.

In [9]:
# Una conversacion de Llama Stack por numero de telefono
conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

#### **GENERAR LA RESPUESTA CON HISTORIAL Y UNA REGLA FIJA**

Se define la lógica que arma cada respuesta: si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento (una decisión que conviene fijar por regla, no dejarla al modelo); en cualquier otro caso, se reenvía todo el historial reciente a Llama, para que pueda resolver referencias a mensajes anteriores.

In [10]:
# Definir la funcion de generacion de respuesta con memoria

FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "agente humano"]

SYSTEM_PROMPT = (
    "Eres el asistente de agendado de citas de una clinica dental. "
    "Responde de forma breve (2-3 lineas)."
)

def generar_respuesta(numero, texto_usuario):
    texto_low = texto_usuario.lower()

    if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
        return "Claro, en un momento un miembro del equipo te contacta directamente."

    conversation_id = obtener_conversacion(numero)

    respuesta = client.responses.create(
        model="ollama/llama3.1:latest",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
    )

    return respuesta.output_text

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [11]:
# Definir endpoint de verificacion de credenciales
@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}

In [12]:
# Definir endpoint de recepcion de mensajes
@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}

In [20]:
# Definir funcion de envio de respuesta
respuestas_enviadas = []
def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v26.0/{PHONE_NUMBER_ID}/messages"

    headers = {
        "Authorization": f"Bearer {WHATSAPP_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)

    respuestas_enviadas.append({
        "numero": numero,
        "status": r.status_code,
        "texto": r.text
    })

In [14]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama
errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()

            #if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                #enviar_respuesta(numero,"Claro, en un momento un miembro del equipo te contacta directamente.")
            #else:
            respuesta = generar_respuesta(numero, texto)
            enviar_respuesta(numero, respuesta)

        except Exception as e:
            errores_procesamiento.append({
                "numero": numero,
                "texto": texto,
                "error": repr(e)
            })

In [15]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [16]:
# Instalar e iniciar tunel publico con ngrok

!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

tunel_publico = ngrok.connect(8000)

print("URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

URL a configurar como Callback URL en el dashboard de Meta:
https://unneeded-playhouse-tuition.ngrok-free.dev/webhook


### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje mencionando el servicio dental, y un segundo mensaje que depende del primero — para confirmar que el historial sí se está reenviando.

In [17]:
# Mandar el primer mensaje, mencionando el servicio

payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215536699933", "text": {"body": "Quiero una cita dental"}}  # 5215500000000
    ]}}]}]
}

r1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_1)
print("Mensaje 1:", r1.status_code, r1.text)

INFO:     127.0.0.1:32838 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: 200 {"status":"ok"}


In [18]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio

payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215536699933", "text": {"body": "¿Tienen horario el sábado?"}}
    ]}}]}]
}

r2 = requests.post("http://127.0.0.1:8000/webhook", json=payload_2)
print("Mensaje 2:", r2.status_code, r2.text)

# La respuesta demora aprox. de 1 a 2 min. al ejecutar local en Colab

INFO:     127.0.0.1:32854 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 2: 200 {"status":"ok"}


In [25]:
numero = "525536699933"

conversaciones_por_numero.pop(numero, None)

print("Conversacion reiniciada")

Conversacion reiniciada


In [26]:
# Verificar respuestas

respuestas_enviadas

[{'numero': '525536699933',
  'status': 401,
  'texto': '{"error":{"message":"Authentication Error","code":190,"type":"OAuthException","fbtrace_id":"AmxRCAfvmZqFHlEz01fB6H6"}}'},
 {'numero': '525536699933',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"525536699933","wa_id":"5215536699933"}],"messages":[{"id":"wamid.HBgNNTIxNTUzNjY5OTkzMxUCABEYEkIzRUM3NDYwMzBGMDBDMDA2RQA="}]}'},
 {'numero': '525536699933',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"525536699933","wa_id":"5215536699933"}],"messages":[{"id":"wamid.HBgNNTIxNTUzNjY5OTkzMxUCABEYEjY5QzQ2NjM2MzQ3Qzg3QzZEOAA="}]}'},
 {'numero': '525536699933',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"525536699933","wa_id":"5215536699933"}],"messages":[{"id":"wamid.HBgNNTIxNTUzNjY5OTkzMxUCABEYEjhCNTE3MDI2NThBNzhBRUQxNAA="}]}'},
 {'numero': '525536699933',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[

In [27]:
# Ver mensajes recibidos
mensajes_recibidos

[{'entry': [{'changes': [{'value': {'messages': [{'from': '5215536699933',
         'text': {'body': 'Quiero una cita dental'}}]}}]}]},
 {'entry': [{'changes': [{'value': {'messages': [{'from': '5215536699933',
         'text': {'body': '¿Tienen horario el sábado?'}}]}}]}]},
 {'object': 'whatsapp_business_account',
  'entry': [{'id': '1389496259965043',
    'changes': [{'value': {'messaging_product': 'whatsapp',
       'metadata': {'display_phone_number': '15551699143',
        'phone_number_id': '1321115231092474'},
       'contacts': [{'wa_id': '5215536699933',
         'user_id': 'MX.1563261751784479'}],
       'statuses': [{'id': 'wamid.HBgNNTIxNTUzNjY5OTkzMxUCABEYEkIzRUM3NDYwMzBGMDBDMDA2RQA=',
         'status': 'sent',
         'timestamp': '1790608186',
         'recipient_id': '5215536699933',
         'recipient_user_id': 'MX.1563261751784479',
         'pricing': {'billable': False,
          'pricing_model': 'PMP',
          'category': 'service',
          'type': 'free_cus

In [28]:
# Verificar almacenamiento en Llama Stack

numero = normalizar_numero_mx("5215536699933")  # tu numero real de WhatsApp
conv_id = conversaciones_por_numero[numero]

items = client.conversations.items.list(conversation_id=conv_id)

for item in items.data:
    print(item.role, "->", item.content)

assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='Es el próximo martes, 14 de marzo. ¿Quieres tomar la cita o prefieres verificar con tu calendario para asegurarte que no tengas otros compromisos?', annotations=[], logprobs=[], type='output_text')]
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='Que martes?', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='La cita de limpieza dental matinal está libre el martes a las 17:00 horas. ¿Te parece esa fecha y hora?', annotations=[], logprobs=[], type='output_text')]
user -> [OpenAIResponseMes

In [ ]:
# Finalizar proceso de tunelización

from pyngrok import ngrok

try:
    ngrok.disconnect(tunel_publico.public_url)
except:
    pass

ngrok.kill()

print("Tunel ngrok finalizado")

## **CHALLENGE: AGENTE DE UNA VETERINARIA**

Una vez visto el ***Hands-On: Diseño de Agentes Conversacionales***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un agente similar, pero para una veterinaria: además de la regla de escalamiento a una persona, se agrega una regla de urgencia médica, y un límite al historial que se conserva en la conversación de Llama Stack.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab, descarga el modelo Llama 3.1 8B, instala y levanta **Llama Stack** con Ollama registrado, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [1]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)

!sudo apt-get install -y zstd >/dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1

In [2]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess, time

ollama = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

!ollama pull llama3.1

# Verificar Ollama y modelo

!ollama list


NAME               ID              SIZE      MODIFIED               
llama3.1:latest    46e0c10c039e    4.9 GB    Less than a second ago    


In [3]:
# Instalar dependencias

!pip install -q fastapi uvicorn requests pyngrok llama-stack-client
!pip install -q "llama-stack[starter]" "mcp<2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 8.3 MB/s eta 0:00:00

In [6]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import os, yaml, llama_stack, subprocess, time

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

config_path = os.path.join(
    os.path.dirname(llama_stack.__file__),
    "distributions", "starter", "config.yaml"
)

with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:latest",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:latest",
    "model_type": "llm"
})

mi_config = "/content/llama_stack.yaml"

with open(mi_config, "w") as f:
    yaml.dump(config, f, sort_keys=False)

proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config, "--port", "8321"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(15)
print("Llama Stack activo:", proceso_llama_stack.poll() is None)

# Verificar Llama Stack

import requests

r = requests.get("http://127.0.0.1:8321/v1/models")
print(r.status_code)
print(r.json()["data"][0]["id"])

# Crear cliente de Llama Stack

from llama_stack_client import LlamaStackClient

client = LlamaStackClient(base_url="http://127.0.0.1:8321")

Llama Stack activo: False
200
ollama/llama3.1:latest


In [33]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

WHATSAPP_TOKEN = userdata.get("WHATSAPP_TOKEN")
PHONE_NUMBER_ID = userdata.get("PHONE_NUMBER_ID")
VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")

print("Credenciales cargadas:", all([
    WHATSAPP_TOKEN,
    PHONE_NUMBER_ID,
    VERIFY_TOKEN,
    NGROK_AUTHTOKEN
]))

Credenciales cargadas: True


**2. Construye la memoria de conversación:** Crea una conversación de **Llama Stack** por número de teléfono (`client.conversations.create()`) con una memoria de respuesta.

In [9]:
# Conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversaciones_por_numero[numero] = client.conversations.create().id
    return conversaciones_por_numero[numero]

In [10]:
# Definir la funcion de generacion de respuesta con memoria

SYSTEM_PROMPT = (
    "Eres el asistente de una veterinaria. "
    "Responde de forma breve, clara y útil."
)

def generar_respuesta(numero, texto):
    respuesta = client.responses.create(
        model="ollama/llama3.1:latest",
        input=texto,
        conversation=obtener_conversacion(numero),
        instructions=SYSTEM_PROMPT
    )
    return respuesta.output_text

In [11]:
print(generar_respuesta("525500000000", "Tengo un perro labrador"))
print(generar_respuesta("525500000000", "¿Qué raza te dije que tengo?"))

¡Excelente raza! ¿En qué puedo ayudarte con tu perro labrador? ¿Tiene problemas de salud, comportamiento o necesitas consejos sobre su cuidado?
Me dijiste que tienes un perro Labrador.


**3. Agrega la regla fija y protege el procesamiento por número:**

   * Cuando el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento.
   * Igual que en el Hands-On, usa un lock por número de teléfono para que dos mensajes seguidos del mismo usuario no disparen dos llamadas en paralelo sobre la misma conversación.

In [12]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario
import threading

locks_por_numero = {}

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

In [13]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama

FRASES_ESCALAMIENTO = [
    "hablar con alguien",
    "hablar con una persona",
    "agente humano"
]

FRASES_URGENCIA = [
    "no respira",
    "sangra mucho",
    "convulsiona",
    "atropellado",
    "envenenado"
]

# Generar respuesta con reglas y memoria

def generar_respuesta(numero, texto):
    texto_low = texto.lower()

    if any(f in texto_low for f in FRASES_ESCALAMIENTO):
        return "Claro, un miembro del equipo te contactará directamente."

    if any(f in texto_low for f in FRASES_URGENCIA):
        return "Es una posible urgencia veterinaria. Acude de inmediato a una clínica veterinaria."

    respuesta = client.responses.create(
        model="ollama/llama3.1:latest",
        input=texto,
        conversation=obtener_conversacion(numero),
        instructions=SYSTEM_PROMPT
    )

    return respuesta.output_text

**4. Define los endpoints del webhook:** usa `BackgroundTasks` para responder de inmediato a Meta, delegando el procesamiento real a la función anterior.

In [14]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests

app = FastAPI()

mensajes_recibidos = []
respuestas_enviadas = []
errores_procesamiento = []


In [15]:
# Definir función de normalización de número
def normalizar_numero_mx(numero):
    return "52" + numero[3:] if numero.startswith("521") and len(numero) == 13 else numero

In [16]:
# Definir endpoint de verificacion de credenciales
@app.get("/webhook")
async def verificar_webhook(request: Request):
    p = request.query_params

    if p.get("hub.verify_token") == VERIFY_TOKEN:
        return int(p.get("hub.challenge"))

    return {"error": "token invalido"}

In [18]:
# Definir endpoint de recepcion de mensajes
@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]

        background_tasks.add_task(procesar_y_responder, numero, texto)

    except (KeyError, IndexError):
        pass

    return {"status": "ok"}

In [19]:
# Definir funcion de envio de respuesta

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v26.0/{PHONE_NUMBER_ID}/messages"

    headers = {
        "Authorization": f"Bearer {WHATSAPP_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)

    respuestas_enviadas.append({
        "numero": numero,
        "status": r.status_code,
        "texto": r.text
    })

    # Procesar mensaje y responder

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            respuesta = generar_respuesta(numero, texto)
            enviar_respuesta(numero, respuesta)

        except Exception as e:
            errores_procesamiento.append({
                "numero": numero,
                "texto": texto,
                "error": repr(e)
            })

**5. Expón el servidor con ngrok y prueba el flujo completo:**

   * Manda una conversación de al menos 4 mensajes y confirma que el historial se recorta correctamente.
   * Manda un mensaje con una palabra de urgencia y confirma que dispara la regla fija en vez de una respuesta de Llama.

In [20]:
# Instalar e iniciar tunel publico con ngrok

from pyngrok import ngrok
import uvicorn, threading

ngrok.set_auth_token(NGROK_AUTHTOKEN)

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

tunel_publico = ngrok.connect(8000)

print("Callback URL:")
print(tunel_publico.public_url + "/webhook")

INFO:     Started server process [1215]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Callback URL:
https://unneeded-playhouse-tuition.ngrok-free.dev/webhook


In [26]:
# Mandar el primer mensaje, mencionando el servicio

payload = {
    "entry": [{
        "changes": [{
            "value": {
                "messages": [{
                    "from": "5215536699933",
                    "text": {"body": "Hola, quiero una cita para mi perro"}
                }]
            }
        }]
    }]
}

r = requests.post("http://127.0.0.1:8000/webhook", json=payload)

print("Webhook:", r.status_code, r.text)

INFO:     127.0.0.1:50472 - "POST /webhook HTTP/1.1" 200 OK
Webhook: 200 {"status":"ok"}


In [27]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio

payload_2 = {
    "entry": [{
        "changes": [{
            "value": {
                "messages": [{
                    "from": "5215536699933",
                    "text": {"body": "¿Tienen horario el sábado?"}
                }]
            }
        }]
    }]
}

r2 = requests.post("http://127.0.0.1:8000/webhook", json=payload_2)

print("Mensaje 2:", r2.status_code, r2.text)

INFO:     127.0.0.1:55888 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 2: 200 {"status":"ok"}


In [28]:
# Verificar respuestas
for r in respuestas_enviadas:
    print(r)

{'numero': '525555555555', 'status': 401, 'texto': '{"error":{"message":"Authentication Error","code":190,"type":"OAuthException","fbtrace_id":"A7ATOfavMiN-vLrjOs5WFTv"}}'}
{'numero': '525555555555', 'status': 401, 'texto': '{"error":{"message":"Authentication Error","code":190,"type":"OAuthException","fbtrace_id":"AYgx09hQ-3mn_f1Doh2fBn-"}}'}
{'numero': '525536699933', 'status': 401, 'texto': '{"error":{"message":"Authentication Error","code":190,"type":"OAuthException","fbtrace_id":"Abxss3BZrQgsOtAwep-fN-y"}}'}
{'numero': '525536699933', 'status': 401, 'texto': '{"error":{"message":"Authentication Error","code":190,"type":"OAuthException","fbtrace_id":"A0l1jHMWN8gcSMyBrD_23gR"}}'}
{'numero': '525536699933', 'status': 401, 'texto': '{"error":{"message":"Authentication Error","code":190,"type":"OAuthException","fbtrace_id":"AaIvVIdRTrkvpiEAOIsVeu3"}}'}


In [29]:
# Ver mensajes recibidos
for m in mensajes_recibidos:
    print(m)

{'entry': [{'changes': [{'value': {'messages': [{'from': '5215555555555', 'text': {'body': 'Hola, quiero una cita para mi perro'}}]}}]}]}
{'entry': [{'changes': [{'value': {'messages': [{'from': '5215555555555', 'text': {'body': '¿Tienen horario el sábado?'}}]}}]}]}
{'object': 'whatsapp_business_account', 'entry': [{'id': '1389496259965043', 'changes': [{'value': {'messaging_product': 'whatsapp', 'metadata': {'display_phone_number': '15551699143', 'phone_number_id': '1321115231092474'}, 'contacts': [{'profile': {'name': 'Eliot Max'}, 'wa_id': '5215536699933', 'user_id': 'MX.1563261751784479'}], 'messages': [{'from': '5215536699933', 'from_user_id': 'MX.1563261751784479', 'id': 'wamid.HBgNNTIxNTUzNjY5OTkzMxUCABIYIEE1OTlGOTQ3NjExQ0ZEMTQxQTFFOUY1NDAxQzY1NkQyAA==', 'timestamp': '1790611508', 'text': {'body': 'Hola'}, 'type': 'text'}]}, 'field': 'messages'}]}]}
{'entry': [{'changes': [{'value': {'messages': [{'from': '5215536699933', 'text': {'body': 'Hola, quiero una cita para mi perro'}}]

In [30]:
# Verificar almacenamiento en Llama Stack

numero = normalizar_numero_mx("5215536699933")
conv_id = conversaciones_por_numero[numero]

items = client.conversations.items.list(conversation_id=conv_id)

for item in items.data:
    print(item.role, "->", item.content)

assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='Sí, estamos abiertos el sábado. Tenemos dos horarios disponibles: mañana a las 9:00 o a las 14:00. ¿Cuál de los dos horarios te funciona mejor?', annotations=[], logprobs=[], type='output_text')]
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='¿Tienen horario el sábado?', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='Claro! ¿Cuándo necesitas la cita? ¿Qué tipo de cita buscas? Examen de rutina, vacunación, consulta de salud o algo más?', annotations=[], logprobs=[], type='output_text')

In [35]:
# Finalizar proceso de tunelización

from pyngrok import ngrok

try:
    ngrok.disconnect(tunel_publico.public_url)
except:
    pass

ngrok.kill()

print("Tunel ngrok finalizado")

Tunel ngrok finalizado
